# Generación de árboles no isomorfos

Se enumeran los árboles de orden $n$ hasta isomorfismo y se calculan la entropía de von Neumann $S(T)$, el primer índice de Zagreb $Z_1(T)$ y el índice de Wiener $W(T)$. Los resultados se almacenan en SQLite.

## 1. Parámetros de ejecución

Se establece el orden de los árboles y el tamaño de los lotes de escritura.

In [ ]:
import os
import sqlite3

import networkx as nx
import numpy as np

NUM_VERTICES = 23
TOL_CERO = 1e-12
PROGRESO_CADA = 100000
TAMANO_LOTE = 10000

CARPETA_SALIDA = f"resultados_seminario_orden_{NUM_VERTICES}"
CARPETA_DATOS = os.path.join(CARPETA_SALIDA, "datos")
os.makedirs(CARPETA_DATOS, exist_ok=True)

print(f"Orden: n={NUM_VERTICES}")
print(f"Directorio de datos: {os.path.abspath(CARPETA_DATOS)}")

## 2. Cálculo de invariantes

Para cada árbol $T$ de orden $n$ se consideran

$$
\rho_T=\frac{L(T)}{2(n-1)},\qquad S(T)=-\sum_{p_i>0}p_i\log_2p_i,
$$

$$
Z_1(T)=\sum_{v\in V(T)}d(v)^2,\qquad W(T)=\sum_{\{u,v\}\subseteq V(T)}d_T(u,v).
$$

El índice de Wiener se calcula sumando $s(n-s)$ sobre las aristas, donde $s$ es el tamaño de una de las componentes al retirar la arista.

In [ ]:
def laplaciano(G):
    """Matriz laplaciana combinatoria: L = D - A."""
    n = G.number_of_nodes()
    L = np.zeros((n, n), dtype=float)

    for v, d in G.degree():
        L[v, v] = d

    for u, v in G.edges():
        L[u, v] = -1.0
        L[v, u] = -1.0

    return L


def espectro_laplaciano(G):
    """Espectro laplaciano ordenado de menor a mayor."""
    valores = np.linalg.eigvalsh(laplaciano(G))
    valores[np.abs(valores) < TOL_CERO] = 0.0
    return valores


def entropia_desde_espectro(espectro, n):
    """Entropía de von Neumann, en bits."""
    normalizacion = 2 * (n - 1)

    p = np.asarray(
        espectro,
        dtype=float
    ) / normalizacion

    p = p[p > TOL_CERO]

    return float(
        -np.sum(p * np.log2(p))
    )


def codigo_arbol(T):
    """Codificación graph6 del árbol."""
    return (
        nx.to_graph6_bytes(
            T,
            header=False
        )
        .decode("ascii")
        .strip()
    )


def indice_wiener_arbol(T):
    """Calcula Wiener en O(n) mediante las contribuciones s(n-s)."""
    n = T.number_of_nodes()

    if n <= 1:
        return 0

    raiz = 0
    padre = [-2] * n
    padre[raiz] = -1
    orden = [raiz]

    for u in orden:
        for v in T[u]:
            if v != padre[u]:
                padre[v] = u
                orden.append(v)

    tamanio = [1] * n
    W = 0

    for v in reversed(orden[1:]):
        s = tamanio[v]
        W += s * (n - s)
        tamanio[padre[v]] += s

    return int(W)


def invariantes_basicos(T):
    """Primer índice de Zagreb e índice de Wiener."""
    zagreb1 = int(
        sum(
            d**2
            for _, d in T.degree()
        )
    )

    return {
        "zagreb1": zagreb1,
        "wiener": indice_wiener_arbol(T),
    }

## 3. Almacenamiento en SQLite

Cada registro contiene `indice`, `codigo`, `entropia`, `zagreb1` y `wiener`. La escritura por lotes permite reanudar enumeraciones interrumpidas.

In [ ]:
def ruta_base_orden(n):
    return os.path.join(
        CARPETA_DATOS,
        f"arboles_orden_{n}.sqlite"
    )


def conectar_base(ruta):
    """Conexión de escritura a la base SQLite."""
    con = sqlite3.connect(ruta)
    con.row_factory = sqlite3.Row

    con.execute("PRAGMA synchronous = NORMAL")
    con.execute("PRAGMA journal_mode = DELETE")
    con.execute("PRAGMA temp_store = FILE")
    con.execute("PRAGMA cache_size = -65536")

    return con


def crear_estructura_base(con, n):
    con.execute("""
        CREATE TABLE IF NOT EXISTS arboles (
            indice INTEGER PRIMARY KEY,
            codigo TEXT NOT NULL,
            entropia REAL NOT NULL,
            zagreb1 INTEGER NOT NULL,
            wiener INTEGER NOT NULL
        )
    """)

    con.execute("""
        CREATE TABLE IF NOT EXISTS metadata (
            clave TEXT PRIMARY KEY,
            valor TEXT NOT NULL
        )
    """)

    fila_n = con.execute(
        "SELECT valor FROM metadata WHERE clave = 'n'"
    ).fetchone()
    if fila_n is not None and fila_n["valor"] != str(n):
        raise ValueError(
            f"Incompatibilidad de orden: base n={fila_n['valor']}, solicitado n={n}."
        )

    con.execute(
        """
        INSERT OR REPLACE INTO metadata(clave, valor)
        VALUES('n', ?)
        """,
        (str(n),)
    )

    con.execute(
        """
        INSERT OR IGNORE INTO metadata(clave, valor)
        VALUES('completa', '0')
        """
    )

    con.commit()


def metadata_base(con, clave, defecto=None):
    fila = con.execute(
        "SELECT valor FROM metadata WHERE clave = ?",
        (clave,)
    ).fetchone()

    return (
        fila["valor"]
        if fila is not None
        else defecto
    )


def crear_indices_base(con):
    print("Construcción de índices SQLite...")

    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_entropia
        ON arboles(entropia, indice)
    """)

    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_zagreb_entropia
        ON arboles(zagreb1, entropia, indice)
    """)

    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_wiener_entropia
        ON arboles(wiener, entropia, indice)
    """)

    con.execute("""
        CREATE INDEX IF NOT EXISTS idx_zagreb_wiener
        ON arboles(zagreb1, wiener)
    """)

    con.commit()


def generar_base_orden(
    n,
    mostrar_progreso=True
):
    ruta = ruta_base_orden(n)
    con = conectar_base(ruta)
    crear_estructura_base(con, n)

    completa = (
        metadata_base(con, "completa", "0")
        == "1"
    )

    total_esperado = nx.number_of_nonisomorphic_trees(n)

    if completa:
        datos = con.execute(
            "SELECT COUNT(*) AS c, MIN(indice) AS minimo, "
            "MAX(indice) AS maximo FROM arboles"
        ).fetchone()
        total = int(datos["c"])
        if (total != total_esperado or datos["minimo"] != 0
                or datos["maximo"] != total_esperado - 1):
            con.close()
            raise RuntimeError(
                f"La base de n={n} figura como completa, pero sus "
                "registros no coinciden con la enumeración esperada."
            )
        con.close()
        print(f"n={n}: base completa ({total:,} árboles).")
        return ruta

    total_actual = int(
        con.execute(
            "SELECT COUNT(*) AS c FROM arboles"
        ).fetchone()["c"]
    )

    fila = con.execute(
        "SELECT MAX(indice) AS m FROM arboles"
    ).fetchone()

    ultimo = fila["m"]

    inicio = (
        int(ultimo) + 1
        if ultimo is not None
        else 0
    )

    # Los índices de la enumeración deben ser consecutivos.
    primer_indice = con.execute(
        "SELECT MIN(indice) AS m FROM arboles"
    ).fetchone()["m"]

    if total_actual != inicio or (total_actual and primer_indice != 0):
        con.close()
        raise RuntimeError(
            f"La base parcial de n={n} no tiene índices consecutivos. "
            "No es posible reanudar la enumeración con índices no consecutivos."
        )

    if total_actual == total_esperado:
        print(
            f"n={n}: registros completos. "
            "Construcción de índices SQLite."
        )

    else:
        if inicio > 0:
            print(
                f"n={n}: reanudación desde el índice "
                f"{inicio:,}."
            )

        lote = []

        for indice, T in enumerate(
            nx.nonisomorphic_trees(n)
        ):
            if indice < inicio:
                continue

            espectro = espectro_laplaciano(T)

            invariantes = invariantes_basicos(T)

            lote.append((
                indice,
                codigo_arbol(T),
                entropia_desde_espectro(
                    espectro,
                    n
                ),
                invariantes["zagreb1"],
                invariantes["wiener"],
            ))

            if len(lote) >= TAMANO_LOTE:
                con.executemany(
                    """
                    INSERT OR REPLACE INTO arboles
                    (indice, codigo, entropia, zagreb1, wiener)
                    VALUES (?, ?, ?, ?, ?)
                    """,
                    lote
                )

                con.commit()
                lote.clear()

            cantidad = indice + 1

            if (
                mostrar_progreso
                and PROGRESO_CADA > 0
                and cantidad % PROGRESO_CADA == 0
            ):
                print(
                    f"n={n}: {cantidad:,} / "
                    f"{total_esperado:,} árboles procesados"
                )

        if lote:
            con.executemany(
                """
                INSERT OR REPLACE INTO arboles
                (indice, codigo, entropia, zagreb1, wiener)
                VALUES (?, ?, ?, ?, ?)
                """,
                lote
            )
            con.commit()

    total = con.execute(
        "SELECT COUNT(*) AS c FROM arboles"
    ).fetchone()["c"]

    if total != total_esperado:
        con.close()
        raise RuntimeError(
            f"La base de n={n} tiene {total:,} filas, "
            f"pero se esperaban {total_esperado:,}."
        )

    crear_indices_base(con)

    con.execute(
        """
        INSERT OR REPLACE INTO metadata(clave, valor)
        VALUES('completa', '1')
        """
    )
    con.commit()
    con.close()

    print(
        f"n={n}: generación completada. "
        f"{total:,} árboles guardados en disco."
    )

    return ruta

## 4. Ejecución

Se genera exclusivamente la base correspondiente al orden seleccionado.

In [ ]:
ruta_db = generar_base_orden(
    NUM_VERTICES,
    mostrar_progreso=True
)

print("Base SQLite:", os.path.abspath(ruta_db))